# 🚀 Chest X-Ray MLOps Pipeline In-Memory (Google Colab TPU V6e Optimized)
Este notebook ejecuta el pipeline de arquitectura desacoplada de Chest X-Ray utilizando la inmensa potencia algorítmica de la nube distribuida (**TPUClusterResolver**). Implementa toda la lógica Focus/GridSearch de MLOps en Memoria asíncrona dentro del Scope de los chips Tensor Processing Units.

In [ ]:
# 1. Montaje Seguro de Infraestructura (Drive Persistencia)
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BACKUP_PATH = '/content/drive/MyDrive/ChestXRayProject'
if not os.path.exists(DRIVE_BACKUP_PATH):
    print(f"⚠️ Directorio de backup {DRIVE_BACKUP_PATH} ausente. Creándolo...")
    os.makedirs(DRIVE_BACKUP_PATH, exist_ok=True)

print("✅ Drive montado y verificado. Persistencia Segura activada.")

In [ ]:
# 2. Clonación Oficial y Setup de Dependencias Médicas
!rm -rf /content/project
!git clone https://github.com/Many871027/Decoupled-Representation-Rebalanced-Classifier-for-X-Rays- /content/project
%cd /content/project

!pip install fastapi uvicorn pydantic scikit-learn pillow pandas tabulate > /dev/null
print("✅ Core MLOps desplegado en el motor Colab Local.")

In [ ]:
# 3. Inyección de MLOps Settings (Distributed TPU Strategy)
import sys
import tensorflow as tf

print("🔄 Escaneando Topología de Clúster de Alto Rendimiento...")
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
    print(f"✅ ¡Hardware Distribuido Acoplado! TPU Cores Enlazados: {strategy.num_replicas_in_sync}")
except ValueError as e:
    print(f"⚠️ Tensor Processing Unit no detectada ({e}). Degradando a Estrategia Single-Device (GPU/CPU).")
    strategy = tf.distribute.get_strategy()

sys.path.append('/content/project')
import src.config as config

config.IMG_HEIGHT = 512
config.IMG_WIDTH = 512
# Matemática Dinámica de Escalado de Chips. x16 para cada chip engranado (Eg. TPUv6 = Batch 128 a 256 simultáneos)
config.BATCH_SIZE = 16 * strategy.num_replicas_in_sync
config.FOCAL_ALPHA = 0.5

DATA_TARGET = '/content/drive/MyDrive/ChestXRayProject/chest_xray'
if os.path.exists(DATA_TARGET):
    config.DATA_DIR = os.path.abspath(DATA_TARGET)
    print(f"✅ Operando distribución Host-Driven para el Dataset: {config.DATA_DIR}")
else:
    print("⚠️ Faltan Imágenes MLOps en Drive. Usando la ruta base clonada.")

In [ ]:
# 4. Generación Dataloaders Optimizados
from src.data_pipeline import get_dataloaders

try:
    train_ds, val_ds, test_ds, class_weights = get_dataloaders({'batch_size': config.BATCH_SIZE})
    print("✅ Dataloaders Médicos compilados asincronamente para ingesta de Clúster.")
except Exception as e:
    print(f"❌ Fatal Error (Revisa Dataset): {e}")

In [ ]:
# 5. Grid Search Memory Workflow (Phase 1 Training sobre TPU HBM)
import itertools
import pandas as pd
from src.model_pipeline import build_custom_cnn_backbone, build_full_model, FocalLoss, MedicalReportCallback

grid_dropouts = [0.4, 0.5]
grid_lr = [1e-3, 5e-4]

experiment_results = []
best_overall_f1 = 0.0
best_model_run = None

for dropout, lr in itertools.product(grid_dropouts, grid_lr):
    print(f"\n{'='*50}\n▶️ Iniciando Iteración (Clúster HBM): Dropout={dropout} | LR={lr}\n{'='*50}")

    # Encapsulamiento forzado dentro de la Matriz del Cluster
    with strategy.scope():
        backbone = build_custom_cnn_backbone()
        model = build_full_model(backbone, dropout_rate=dropout)
        optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
        model.compile(optimizer=optimizer, loss=FocalLoss(), metrics=['accuracy'])

    medical_report = MedicalReportCallback(val_ds)

    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_f1_score',
        mode='max',
        patience=8,
        restore_best_weights=True,
        verbose=1
    )
    lr_reducer = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, verbose=1)

    # Distribución Tensor-Level
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=50,
        callbacks=[lr_reducer, medical_report, early_stopping],
        verbose=1
    )

    val_acc = max(history.history.get('val_accuracy', [0]))
    val_f1 = max(history.history.get('val_f1_score', [0]))

    is_best_run = False
    if val_f1 > best_overall_f1:
        best_overall_f1 = val_f1
        best_model_run = model
        is_best_run = True
        print(f"⭐ ¡Nuevo Vértice MLOps! Macro-F1 Local: {val_f1:.4f}")

    experiment_results.append({
        'Dropout': dropout,
        'LR': lr,
        'Mem. Cores': strategy.num_replicas_in_sync,
        'Opt. Batch Size': config.BATCH_SIZE,
        'Best F1 (Macro)': round(val_f1, 4),
        'Best Acc.': round(val_acc, 4),
        'Focal Alpha': config.FOCAL_ALPHA,
        'Modelo Elite': is_best_run
    })

print("✅ Entrenamiento Tensor Multicore Finalizado.")

## 🏆 Reporte Tabular y Backup Físico a Google Drive
Se imprimirán los resultados de todos los tests renderizados y se exportará el modelo TPU absoluto (.keras) evitando cualquier pérdida o evaporación de máquina virtual.

In [ ]:
# 6. Extracción de Métricas Pandas & Drive Auto-Save
import pandas as pd
from IPython.display import display

df_results = pd.DataFrame(experiment_results)
df_results = df_results.sort_values(by='Best F1 (Macro)', ascending=False).reset_index(drop=True)

print("\n" + "*"*60)
print("🏔️ CLÚSTER ANALYSIS: HIPERPARÁMETROS EXPERIMENTALES TPU 🏔️")
print("*"*60 + "\n")

display(df_results.style.background_gradient(cmap='Blues', subset=['Best F1 (Macro)']))

if best_model_run:
    # Extraemos el peso del TPU de regreso a la RAM principal para serializarlo en disco
    TARGET_SAVE = f"{DRIVE_BACKUP_PATH}/Model_TPU_Elite_F1_{best_overall_f1:.4f}.keras"
    print("\n⚙️ Copiando y exportando pesos compilados al entorno persistente Drive...")
    best_model_run.save(TARGET_SAVE)
    print(f"✅ EXPORTACIÓN TPU EXITOSA: {TARGET_SAVE}")
else:
    print("❌ Fallo crítico. No hay modelo para persistir en Disco.")